# Spark Performance Optimization Experiments

This notebook demonstrates advanced Spark engineering techniques:

• Broadcast Join Optimization
• Shuffle Analysis
• Data Skew Simulation
• Salting Technique
• Partition Tuning
• Execution Plan Analysis

- 1️⃣ Execution Plan Analysis
- 2️⃣ Broadcast Join Optimization
- 3️⃣ Shuffle & Exchange Deep Dive
- 4️⃣ Data Skew Simulation
- 5️⃣ Skew Join Fix (Salting)
- 6️⃣ Partition Tuning
- 7️⃣ Shuffle Partition Optimization
- 8️⃣ Caching & Persistence
- 9️⃣ AQE Runtime Optimization
- 🔟 Large Dataset Stress Testing

## Experiment 1 — Broadcast Join Optimization

### Objective

The goal of this experiment is to understand how **Spark chooses a join strategy** when joining a very large table with a relatively small table.
We will analyze how Spark’s **Catalyst Optimizer** selects the most efficient join type and how this affects performance.

---

### Scenario

We will join the following tables from the Gold layer:

* **fact_sales** → ~24,000,000 rows (large fact table)
* **dim_product** → ~50,000 rows (small dimension table)

Typical analytical query:

```
fact_sales JOIN dim_product
ON product_id
```

Since **dim_product is significantly smaller**, Spark should automatically choose a **Broadcast Hash Join**.

---

### What is Broadcast Join?

In a broadcast join, Spark **replicates the smaller table to all executors** instead of shuffling both tables across the cluster.

Instead of doing this:

```
fact_sales → shuffle
dim_product → shuffle
```

Spark performs:

```
broadcast(dim_product)
```

Each executor processing a partition of `fact_sales` receives a local copy of `dim_product`.

Execution becomes:

```
Executor 1 → fact_sales partition + dim_product
Executor 2 → fact_sales partition + dim_product
Executor 3 → fact_sales partition + dim_product
```

This eliminates the need for **expensive shuffle operations**.

---

### Why Broadcast Join is Faster

Shuffle operations involve:

* Network data transfer
* Disk spilling
* Serialization and deserialization
* Additional stage boundaries in Spark

Broadcast joins avoid these costs by **sending the small dataset once to each executor**.

---

### Spark Configuration Behind Broadcast

Spark decides automatically whether a table can be broadcast based on a threshold:

```
spark.sql.autoBroadcastJoinThreshold
```

Default value:

```
10 MB
```

If a table is smaller than this threshold, Spark may broadcast it automatically.

---

### What We Will Observe

During this experiment we will inspect the Spark **execution plan** using:

```
df.explain("formatted")
```

We expect to see:

```
BroadcastHashJoin
BroadcastExchange
```

These operators confirm that Spark chose a **broadcast join strategy**.

---

### Key Learning Outcomes

By the end of this experiment we will understand:

* How Spark decides between **broadcast join and shuffle join**
* The role of **Catalyst Optimizer** in query planning
* How to read a **Spark execution plan**
* How broadcast joins reduce **shuffle cost and improve performance**


In [3]:
## First Code Cell (Load Tables) 
## Experiment 1 — Broadcast Join Optimization

fact_sales_df = spark.table("fact_sales")
dim_product_df = spark.table("dim_product")

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 5, Finished, Available, Finished, False)

In [4]:
# Check row counts / Before optimization we always check dataset size.

print("Fact Sales Rows:", fact_sales_df.count())
print("Dim Product Rows:", dim_product_df.count())

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 6, Finished, Available, Finished, False)

Fact Sales Rows: 24000000
Dim Product Rows: 50000


In [5]:
# Now we create the join DataFrame.
# Join fact and dimension tables

join_df = fact_sales_df.join(
    dim_product_df,
    fact_sales_df.product_id == dim_product_df.product_id,
    "inner"
).select(
    fact_sales_df.order_id,
    fact_sales_df.customer_id,
    fact_sales_df.product_id,
    dim_product_df.category,
    fact_sales_df.line_revenue
)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 7, Finished, Available, Finished, False)

In [6]:
# Inspect Spark execution plan

join_df.explain("formatted")

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 8, Finished, Available, Finished, False)

== Physical Plan ==
AdaptiveSparkPlan (6)
+- Project (5)
   +- BroadcastHashJoin Inner BuildRight (4)
      :- Scan parquet spark_catalog.chimcobldhq2ap9dcdnmqrb5e9hmabb6c5h74qb35lo74rraclhn89bcd1fmaorfdlmmasj3clfmqpb4c5m6oqbfdoim8ojf.fact_sales (1)
      +- BroadcastExchange (3)
         +- Scan parquet spark_catalog.chimcobldhq2ap9dcdnmqrb5e9hmabb6c5h74qb35lo74rraclhn89bcd1fmaorfdlmmasj3clfmqpb4c5m6oqbfdoim8ojf.dim_product (2)


(1) Scan parquet spark_catalog.chimcobldhq2ap9dcdnmqrb5e9hmabb6c5h74qb35lo74rraclhn89bcd1fmaorfdlmmasj3clfmqpb4c5m6oqbfdoim8ojf.fact_sales
Output [4]: [order_id#756L, customer_id#757L, product_id#758L, line_revenue#761]
Batched: true
Location: PreparedDeltaFileIndex [abfss://eaf0c26f-0ff8-484b-907a-fbe7a597851a@onelake.dfs.fabric.microsoft.com/8702ce55-bae0-44d5-b9b0-0018d638c4c0/Tables/dbo/fact_sales]
ReadSchema: struct<order_id:bigint,customer_id:bigint,product_id:bigint,line_revenue:double>

(2) Scan parquet spark_catalog.chimcobldhq2ap9dcdnmqrb5e9hmabb6c5

###### **<mark>Experiment 2 — Force SortMergeJoin (Disable Broadcast)
###### <mark>Goal</mark></mark>**

In the previous experiment Spark used: BroadcastHashJoin because dim_product is small.

Now we will disable broadcast join, forcing Spark to use:
SortMergeJoin

This will introduce:

Shuffle
Exchange
Stage boundaries

In [7]:
# Disable automatic broadcast joins

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 9, Finished, Available, Finished, False)

Run the Same Join Again

In [8]:
join_df_no_broadcast = fact_sales_df.join(
    dim_product_df,
    fact_sales_df.product_id == dim_product_df.product_id,
    "inner"
).select(
    fact_sales_df.order_id,
    fact_sales_df.customer_id,
    fact_sales_df.product_id,
    dim_product_df.category,
    fact_sales_df.line_revenue
)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 10, Finished, Available, Finished, False)

In [9]:
join_df_no_broadcast.explain("formatted")

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 11, Finished, Available, Finished, False)

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Project (8)
   +- SortMergeJoin Inner (7)
      :- Sort (3)
      :  +- Exchange (2)
      :     +- Scan parquet spark_catalog.chimcobldhq2ap9dcdnmqrb5e9hmabb6c5h74qb35lo74rraclhn89bcd1fmaorfdlmmasj3clfmqpb4c5m6oqbfdoim8ojf.fact_sales (1)
      +- Sort (6)
         +- Exchange (5)
            +- Scan parquet spark_catalog.chimcobldhq2ap9dcdnmqrb5e9hmabb6c5h74qb35lo74rraclhn89bcd1fmaorfdlmmasj3clfmqpb4c5m6oqbfdoim8ojf.dim_product (4)


(1) Scan parquet spark_catalog.chimcobldhq2ap9dcdnmqrb5e9hmabb6c5h74qb35lo74rraclhn89bcd1fmaorfdlmmasj3clfmqpb4c5m6oqbfdoim8ojf.fact_sales
Output [4]: [order_id#756L, customer_id#757L, product_id#758L, line_revenue#761]
Batched: true
Location: PreparedDeltaFileIndex [abfss://eaf0c26f-0ff8-484b-907a-fbe7a597851a@onelake.dfs.fabric.microsoft.com/8702ce55-bae0-44d5-b9b0-0018d638c4c0/Tables/dbo/fact_sales]
ReadSchema: struct<order_id:bigint,customer_id:bigint,product_id:bigint,line_revenue:double>

(2) Exchange
Inp

###### **<mark>Experiment 3 — Data Skew (Concept First)</mark>**

In [10]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 12, Finished, Available, Finished, False)

In [11]:
join_df_no_broadcast.explain("formatted")

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 13, Finished, Available, Finished, False)

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Project (8)
   +- SortMergeJoin Inner (7)
      :- Sort (3)
      :  +- Exchange (2)
      :     +- Scan parquet spark_catalog.chimcobldhq2ap9dcdnmqrb5e9hmabb6c5h74qb35lo74rraclhn89bcd1fmaorfdlmmasj3clfmqpb4c5m6oqbfdoim8ojf.fact_sales (1)
      +- Sort (6)
         +- Exchange (5)
            +- Scan parquet spark_catalog.chimcobldhq2ap9dcdnmqrb5e9hmabb6c5h74qb35lo74rraclhn89bcd1fmaorfdlmmasj3clfmqpb4c5m6oqbfdoim8ojf.dim_product (4)


(1) Scan parquet spark_catalog.chimcobldhq2ap9dcdnmqrb5e9hmabb6c5h74qb35lo74rraclhn89bcd1fmaorfdlmmasj3clfmqpb4c5m6oqbfdoim8ojf.fact_sales
Output [4]: [order_id#756L, customer_id#757L, product_id#758L, line_revenue#761]
Batched: true
Location: PreparedDeltaFileIndex [abfss://eaf0c26f-0ff8-484b-907a-fbe7a597851a@onelake.dfs.fabric.microsoft.com/8702ce55-bae0-44d5-b9b0-0018d638c4c0/Tables/dbo/fact_sales]
ReadSchema: struct<order_id:bigint,customer_id:bigint,product_id:bigint,line_revenue:double>

(2) Exchange
Inp

In [12]:
# Experiment 3 — Data Skew Simulation

from pyspark.sql.functions import when

# Create skewed fact table
skewed_fact_df = fact_sales_df.withColumn(
    "skewed_product_id",
    when(fact_sales_df.product_id < 100, 1)  # force skew
    .otherwise(fact_sales_df.product_id)
)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 14, Finished, Available, Finished, False)

In [13]:
skewed_fact_df.groupBy("skewed_product_id").count().orderBy("count", ascending=False).show(10)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 15, Finished, Available, Finished, False)

+-----------------+-----+
|skewed_product_id|count|
+-----------------+-----+
|                1|47414|
|            46294|  573|
|            27719|  572|
|             1726|  568|
|            15787|  565|
|            33208|  563|
|            37071|  561|
|             6749|  560|
|            26874|  560|
|            31678|  560|
+-----------------+-----+
only showing top 10 rows



In [14]:
skew_join_df = skewed_fact_df.join(
    dim_product_df,
    skewed_fact_df.skewed_product_id == dim_product_df.product_id,
    "inner"
)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 16, Finished, Available, Finished, False)

In [15]:
skew_join_df.explain("formatted")

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 17, Finished, Available, Finished, False)

== Physical Plan ==
AdaptiveSparkPlan (10)
+- SortMergeJoin Inner (9)
   :- Sort (5)
   :  +- Exchange (4)
   :     +- Project (3)
   :        +- Filter (2)
   :           +- Scan parquet spark_catalog.chimcobldhq2ap9dcdnmqrb5e9hmabb6c5h74qb35lo74rraclhn89bcd1fmaorfdlmmasj3clfmqpb4c5m6oqbfdoim8ojf.fact_sales (1)
   +- Sort (8)
      +- Exchange (7)
         +- Scan parquet spark_catalog.chimcobldhq2ap9dcdnmqrb5e9hmabb6c5h74qb35lo74rraclhn89bcd1fmaorfdlmmasj3clfmqpb4c5m6oqbfdoim8ojf.dim_product (6)


(1) Scan parquet spark_catalog.chimcobldhq2ap9dcdnmqrb5e9hmabb6c5h74qb35lo74rraclhn89bcd1fmaorfdlmmasj3clfmqpb4c5m6oqbfdoim8ojf.fact_sales
Output [6]: [order_id#756L, customer_id#757L, product_id#758L, order_date#759, quantity#760L, line_revenue#761]
Batched: true
Location: PreparedDeltaFileIndex [abfss://eaf0c26f-0ff8-484b-907a-fbe7a597851a@onelake.dfs.fabric.microsoft.com/8702ce55-bae0-44d5-b9b0-0018d638c4c0/Tables/dbo/fact_sales]
ReadSchema: struct<order_id:bigint,customer_id:bigint,prod

In [16]:
skew_join_df.count()

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 18, Finished, Available, Finished, False)

24000000

###### **<mark>Step 1 — Add Salt (Fact Table)</mark>**

In [17]:
from pyspark.sql.functions import rand, floor, concat_ws, col

salted_fact_df = skewed_fact_df.withColumn(
    "salt",
    floor(rand() * 5)
)

salted_fact_df = salted_fact_df.withColumn(
    "salted_key",
    concat_ws("_", col("skewed_product_id"), col("salt"))
)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 19, Finished, Available, Finished, False)

###### **<mark>Step 2 — Expand Dimension</mark>**

In [18]:
from pyspark.sql.functions import explode, array, lit, concat_ws, col

salt_values = [lit(i) for i in range(5)]

expanded_dim_df = dim_product_df.withColumn(
    "salt",
    explode(array(*salt_values))
)

expanded_dim_df = expanded_dim_df.withColumn(
    "salted_key",
    concat_ws("_", col("product_id"), col("salt"))
)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 20, Finished, Available, Finished, False)

###### **<mark>Step 3 — Salted Join</mark>**

In [19]:
salted_join_df = salted_fact_df.join(
    expanded_dim_df,
    salted_fact_df.salted_key == expanded_dim_df.salted_key,
    "inner"
)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 21, Finished, Available, Finished, False)

###### **<mark>Step 4 — Explain Plan</mark>**

In [20]:
salted_join_df.explain("formatted")

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 22, Finished, Available, Finished, False)

== Physical Plan ==
AdaptiveSparkPlan (12)
+- SortMergeJoin Inner (11)
   :- Sort (5)
   :  +- Exchange (4)
   :     +- Project (3)
   :        +- Project (2)
   :           +- Scan parquet spark_catalog.chimcobldhq2ap9dcdnmqrb5e9hmabb6c5h74qb35lo74rraclhn89bcd1fmaorfdlmmasj3clfmqpb4c5m6oqbfdoim8ojf.fact_sales (1)
   +- Sort (10)
      +- Exchange (9)
         +- Project (8)
            +- Generate (7)
               +- Scan parquet spark_catalog.chimcobldhq2ap9dcdnmqrb5e9hmabb6c5h74qb35lo74rraclhn89bcd1fmaorfdlmmasj3clfmqpb4c5m6oqbfdoim8ojf.dim_product (6)


(1) Scan parquet spark_catalog.chimcobldhq2ap9dcdnmqrb5e9hmabb6c5h74qb35lo74rraclhn89bcd1fmaorfdlmmasj3clfmqpb4c5m6oqbfdoim8ojf.fact_sales
Output [6]: [order_id#756L, customer_id#757L, product_id#758L, order_date#759, quantity#760L, line_revenue#761]
Batched: true
Location: PreparedDeltaFileIndex [abfss://eaf0c26f-0ff8-484b-907a-fbe7a597851a@onelake.dfs.fabric.microsoft.com/8702ce55-bae0-44d5-b9b0-0018d638c4c0/Tables/dbo/fact_sale

###### **<mark>Step 5 — Trigger Execution</mark>**

In [21]:
salted_join_df.count()

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 23, Finished, Available, Finished, False)

24000000

###### **<mark>Validate Salted Columns</mark>**

In [22]:
salted_fact_df.select("skewed_product_id", "salt", "salted_key").show(10, False)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 24, Finished, Available, Finished, False)

+-----------------+----+----------+
|skewed_product_id|salt|salted_key|
+-----------------+----+----------+
|21296            |3   |21296_3   |
|42820            |2   |42820_2   |
|8198             |3   |8198_3    |
|7706             |2   |7706_2    |
|46069            |4   |46069_4   |
|43027            |1   |43027_1   |
|46993            |2   |46993_2   |
|18593            |2   |18593_2   |
|14899            |2   |14899_2   |
|16613            |1   |16613_1   |
+-----------------+----+----------+
only showing top 10 rows



###### **<mark>Validate Distribution</mark>**

In [23]:
salted_fact_df.groupBy("salt").count().show()

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 25, Finished, Available, Finished, False)

+----+-------+
|salt|  count|
+----+-------+
|   0|4797721|
|   1|4799140|
|   3|4804727|
|   2|4797260|
|   4|4801152|
+----+-------+



###### **<mark>Validate Skew is Fixed (Critical)</mark>**

In [24]:
salted_fact_df.groupBy("salted_key").count().orderBy("count", ascending=False).show(10, False)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 26, Finished, Available, Finished, False)

+----------+-----+
|salted_key|count|
+----------+-----+
|1_1       |9662 |
|1_0       |9454 |
|1_3       |9450 |
|1_4       |9433 |
|1_2       |9415 |
|30934_2   |148  |
|7449_2    |144  |
|38401_0   |142  |
|41109_4   |140  |
|21584_0   |139  |
+----------+-----+
only showing top 10 rows



###### **<mark>Validate Parallelism (MOST IMPORTANT)</mark>**

###### **<mark>Step 1 Check Number of Partitions</mark>**

In [25]:
print(salted_join_df.rdd.getNumPartitions())

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 27, Finished, Available, Finished, False)

17


###### **<mark>2. Check Data Distribution Per Partition (IMPORTANT 🔥)</mark>**

In [26]:
from pyspark.sql.functions import spark_partition_id

salted_join_df_with_partition = salted_join_df.withColumn(
    "partition_id",
    spark_partition_id()
)

salted_join_df_with_partition.groupBy("partition_id") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(20)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 28, Finished, Available, Finished, False)

+------------+-------+
|partition_id|  count|
+------------+-------+
|           7|2994629|
|           2|2990339|
|           5|2987583|
|           0|2982620|
|           1|2902776|
|           6|2898650|
|           3|2886861|
|           4|2873384|
|           8| 483158|
+------------+-------+



###### **<mark>👉 PERFORMANCE COMPARISON</mark>**

###### **<mark>Step 1 — Run WITHOUT salting</mark>**

In [27]:
normal_join_df = skewed_fact_df.join(
    dim_product_df,
    skewed_fact_df.skewed_product_id == dim_product_df.product_id,
    "inner"
)

normal_join_df.count()

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 29, Finished, Available, Finished, False)

24000000

###### **<mark>Step 2 - Run Without Salting</mark>**

In [28]:
salted_join_df.count()

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 30, Finished, Available, Finished, False)

24000000

###### **<mark>👉 CACHING (next optimization)</mark>**

###### **<mark>Apply Config + cache</mark>**

In [29]:
spark.conf.set("spark.sql.shuffle.partitions", 20)

from pyspark import StorageLevel
salted_join_df.persist(StorageLevel.MEMORY_AND_DISK)
salted_join_df.count()

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 31, Finished, Available, Finished, False)

24000000

###### **<mark>Verify Cache is Working</mark>**

In [30]:
salted_join_df.explain()

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 32, Finished, Available, Finished, False)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   *(5) SortMergeJoin [salted_key#2137], [salted_key#2156], Inner
   :- *(3) Sort [salted_key#2137 ASC NULLS FIRST], false, 0
   :  +- AQEShuffleRead coalesced
   :     +- ShuffleQueryStage 0
   :        +- Exchange hashpartitioning(salted_key#2137, 200), ENSURE_REQUIREMENTS, [plan_id=1465]
   :           +- *(1) Project [order_id#756L, customer_id#757L, product_id#758L, order_date#759, quantity#760L, line_revenue#761, skewed_product_id#1972L, salt#2128L, concat_ws(_, cast(skewed_product_id#1972L as string), cast(salt#2128L as string)) AS salted_key#2137]
   :              +- *(1) Project [order_id#756L, customer_id#757L, product_id#758L, order_date#759, quantity#760L, line_revenue#761, CASE WHEN (product_id#758L < 100) THEN 1 ELSE product_id#758L END AS skewed_product_id#1972L, FLOOR((rand(-9055350986870532637) * 5.0)) AS salt#2128L]
   :                 +- *(1) ColumnarToRow
   :                    +- FileScan

###### **<mark>REPARTITION EXPERIMENT</mark>**

In [31]:
print("Current partitions:", salted_join_df.rdd.getNumPartitions())

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 33, Finished, Available, Finished, False)

Current partitions: 17


###### **<mark>STEP 2: REPARTITION (CONTROL PARALLELISM)</mark>**

In [32]:
df_repartitioned = salted_join_df.repartition(
    40,  # target partitions (we will tune later)
    "customer_id"
)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 34, Finished, Available, Finished, False)

###### **<mark>STEP 3: VERIFY PARTITIONS</mark>**

In [33]:
print("After repartition:", df_repartitioned.rdd.getNumPartitions())

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 35, Finished, Available, Finished, False)

After repartition: 40


###### **<mark>STEP 4: PARTITION DISTRIBUTION CHECK</mark>**

In [34]:
from pyspark.sql.functions import spark_partition_id, count

df_partition_check = (
    df_repartitioned
    .withColumn("partition_id", spark_partition_id())
    .groupBy("partition_id")
    .agg(count("*").alias("record_count"))
    .orderBy("partition_id")
)

display(df_partition_check)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 36, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 34c63e24-4d88-4a12-bbd7-74bd8d8a1cab)

###### **<mark>STEP 5: REPARTITION TO 128 (OPTIMAL) for F64 Capacity </mark>**

In [35]:
df_repartitioned = salted_join_df.repartition(
    128,
    "customer_id"
)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 37, Finished, Available, Finished, False)

###### **<mark>Verify Number of Partition</mark>**

In [36]:
print("After repartition:", df_repartitioned.rdd.getNumPartitions())

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 38, Finished, Available, Finished, False)

After repartition: 128


###### **<mark>Distribution Check</mark>**

In [37]:
from pyspark.sql.functions import spark_partition_id, count

df_partition_check = (
    df_repartitioned
    .withColumn("partition_id", spark_partition_id())
    .groupBy("partition_id")
    .agg(count("*").alias("record_count"))
    .orderBy("partition_id")
)

display(df_partition_check)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 39, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f76c9985-d54e-41dc-b7af-ae3eb6dec32b)

###### **<mark>REPARTITION TUNING DEEP DIVE (CRITICAL IN REAL WORLD)</mark>**

###### **<mark>PROBLEM: SMALL FILE ISSUE</mark>**
When you do:

df_repartitioned.write.format("delta").save(...)

👉 Spark will create:

= equal number of partitions → number of output files

**So -- >>  128 partitions → 128 files** ❗

###### **<mark>🧪 STEP 6: FILE SIZE STRATEGY</mark>**

**<mark>OPTION 1: COALESCE BEFORE WRITE (BEST PRACTICE)</mark>**
**<mark>WHY COALESCE?</mark>**

- ✔ Reduces partitions
- ✔ Avoids full shuffle
- ✔ Optimized for write

In [38]:
df_final_write = df_repartitioned.coalesce(32)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 40, Finished, Available, Finished, False)

**<mark>Validate Partition size</mark>**

In [39]:
print(df_final_write.rdd.getNumPartitions())

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 41, Finished, Available, Finished, False)

32


###### **<mark>WRITE TO DELTA</mark>**

In [40]:
df_final_write.write \
    .format("delta") \
    .mode("overwrite") \
    .save("Tables/gold_orders_optimized")

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 42, Finished, Available, Finished, False)

AnalysisException: [DELTA_DUPLICATE_COLUMNS_FOUND] Found duplicate column(s) in the data to save: product_id, salt, salted_key

**<mark>DROP DUPLICATES (QUICK FIX) - came up with the below error while writing to delta</mark>**

- [DELTA_DUPLICATE_COLUMNS_FOUND] Found duplicate column(s) in the data to save: product_id, salt, salted_key

**Fix applied to overcome this situation**

In [41]:
df_clean = df_final_write.drop("salt", "salted_key", "product_id")

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 43, Finished, Available, Finished, False)

###### <mark>**Write to Delta**</mark>

In [42]:
df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .save("Tables/gold_orders_optimized")

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 44, Finished, Available, Finished, False)

AnalysisException: [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: e7b19b79-7cc3-4a00-8d14-c8c9f0b591c1).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- order_id: long (nullable = true)
-- customer_id: long (nullable = true)
-- order_date: date (nullable = true)
-- quantity: long (nullable = true)
-- line_revenue: double (nullable = true)
-- product_id: long (nullable = true)
-- product_name: string (nullable = true)
-- category: string (nullable = true)
-- price: double (nullable = true)
-- cost: double (nullable = true)
-- product_description: string (nullable = true)


Data schema:
root
-- order_id: long (nullable = true)
-- customer_id: long (nullable = true)
-- order_d

###### **<mark>Product_id accidentally deleted and i'm trying to recover it from a skewed dataframe - this is a good learning of recovering an accidental deletion column</mark>**

**<mark>verified schema of df_clean Dataframe - product_id is missing(means deleted)</mark>**

root
 - |-- order_id: long (nullable = true)
 - |-- customer_id: long (nullable = true)
 - |-- order_date: date (nullable = true)
 - |-- quantity: long (nullable = true)
 - |-- line_revenue: double (nullable = true)
 - |-- skewed_product_id: long (nullable = true)
 - |-- product_name: string (nullable = true)
 - |-- category: string (nullable = true)
 - |-- price: double (nullable = true)
 - |-- cost: double (nullable = true)
 - |-- product_description: string (nullable = true)

In [43]:
df_clean.printSchema()

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 45, Finished, Available, Finished, False)

root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: long (nullable = true)
 |-- line_revenue: double (nullable = true)
 |-- skewed_product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- cost: double (nullable = true)
 |-- product_description: string (nullable = true)



**<mark>STEP 1: RECOVER product_id</mark>**

In [44]:
df_fixed = df_clean.withColumnRenamed(
    "skewed_product_id",
    "product_id"
)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 46, Finished, Available, Finished, False)

**<mark>Verify Schema</mark>**

In [45]:
df_fixed.printSchema()

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 47, Finished, Available, Finished, False)

root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: long (nullable = true)
 |-- line_revenue: double (nullable = true)
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- cost: double (nullable = true)
 |-- product_description: string (nullable = true)



In [46]:
display(df_fixed.select("product_id").limit(2))

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 48, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, aabb1448-68aa-4b60-aa2a-627c3b55b9d7)

**<mark>Write to Delta</mark>**

In [47]:
df_fixed.write \
    .format("delta") \
    .mode("overwrite") \
    .save("Tables/gold_orders_optimized")

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 49, Finished, Available, Finished, False)

###### <mark>**while writing to Delta i came accross below error ( Schema mismatch detected when writing to Delta table )**</mark>

**ROOT CAUSE (VERY IMPORTANT)**

- 👉 <mark>Earlier you successfully wrote df_clean (with wrong schema)</mark>

1. Now: Table already exists ❗
2. Schema changed ❗ 👉 <mark>Delta is schema-enforced (ACID)</mark>



**<mark>Now as a Fix trying to overwrite Schema</mark>**

In [48]:
df_fixed.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("Tables/gold_orders_optimized")

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 50, Finished, Available, Finished, False)

###### **<mark>OOM SIMULATION (HANDS-ON)</mark>**

BAD PATTERN (FOR OOM SIMULATION)

👉 We intentionally: Avoid broadcast

Use large join

Reduce partitions (big partitions = memory pressure)

In [49]:
from pyspark.sql.functions import rand

# Create larger dataset by duplicating data
df_orders_large = salted_join_df.withColumn("dup_key", (rand() * 10).cast("int"))

df_orders_large = df_orders_large.union(df_orders_large)  # double data

print("Row count:", df_orders_large.count())
print("Partitions:", df_orders_large.rdd.getNumPartitions())

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 51, Finished, Available, Finished, False)

Row count: 48000000
Partitions: 40


###### **<mark>STEP 2: FORCE BAD PARTITIONING</mark>**

**👉 Reduce partitions (this is dangerous in real-world)**

In [50]:
df_orders_bad = df_orders_large.coalesce(8)
print("Partitions after coalesce:", df_orders_bad.rdd.getNumPartitions())

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 52, Finished, Available, Finished, False)

Partitions after coalesce: 8


###### <mark>**STEP 3: HEAVY JOIN (OOM TRIGGER)**</mark>

**This creates
Shuffle + Large partitions + Self join = heavy memory usage**

In [51]:
df_join_bad = df_orders_bad.join(
    df_orders_bad,
    "customer_id"
)

# Trigger execution
df_join_bad.count()

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 53, Finished, Available, Finished, False)

2592512064

###### **<mark>PHASE: OOM FIX (STEP-BY-STEP)</mark>**

**<mark>FIX 1: REPARTITION (PRIMARY FIX)</mark>**

In [52]:
df_orders_fixed = df_orders_large.repartition(128, "customer_id")

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 54, Finished, Available, Finished, False)

**<mark>FIX 2: AVOID SELF JOIN EXPLOSION</mark>**
**❌ Problem
df.join(df)**

**👉 Causes: Data multiplication (N × N) ❌**

Safer Alternative (if needed)

👉 Always ask:

**Do I really need self join?**

👉 If yes:

- **Reduce data first**
- **Filter early**
- **Limit dataset**

**<mark>FIX 3: FILTER BEFORE JOIN (VERY IMPORTANT)</mark>**

In [53]:
df_filtered = df_orders_large.filter("quantity > 1")

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 55, Finished, Available, Finished, False)

👉 **Now join:**
**Less data → less shuffle → less memory usage ✅**

In [54]:
df_join_fixed = df_filtered.join(df_filtered, "customer_id")

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 56, Finished, Available, Finished, False)

**<mark>FIX 4: USE BROADCAST (WHEN APPLICABLE)</mark>**

In [55]:
from pyspark.sql.functions import broadcast

df_join = df_orders_large.join(
    broadcast(df_products),
    "product_id"
)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 57, Finished, Available, Finished, False)

NameError: name 'df_products' is not defined

In [ ]:
print(dir())

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, -1, Cancelled, , Cancelled, True)

In [58]:
from pyspark.sql.functions import broadcast

df_join = df_orders_large.join(
    broadcast(dim_product_df),
    "product_id"
)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 60, Finished, Available, Finished, False)

In [ ]:
[v for v in dir() if "df" in v]

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, -1, Cancelled, , Cancelled, True)

In [59]:
df_orders_fixed = df_orders_large.repartition(128, "customer_id")

df_join_fixed = df_orders_fixed.join(df_orders_fixed, "customer_id")

df_join_fixed.count()

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 61, Finished, Available, Finished, False)

2592512064

In [67]:
spark.sql("DROP TABLE IF EXISTS orders")

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 69, Finished, Available, Finished, False)

DataFrame[]

In [75]:
[v for v in dir() if "df" in v]

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 77, Finished, Available, Finished, False)

['df_clean',
 'df_filtered',
 'df_final_write',
 'df_fixed',
 'df_join',
 'df_join_bad',
 'df_join_fixed',
 'df_orders_bad',
 'df_orders_fixed',
 'df_orders_large',
 'df_partition_check',
 'df_repartitioned',
 'dim_product_df',
 'expanded_dim_df',
 'fact_sales_df',
 'join_df',
 'join_df_no_broadcast',
 'normal_join_df',
 'salted_fact_df',
 'salted_join_df',
 'salted_join_df_with_partition',
 'skew_join_df',
 'skewed_fact_df']

In [76]:
print("df_clean columns:", df_clean.columns)
print("df_fixed columns:", df_fixed.columns)
print("df_orders_large columns:", df_orders_large.columns)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 78, Finished, Available, Finished, False)

df_clean columns: ['order_id', 'customer_id', 'order_date', 'quantity', 'line_revenue', 'skewed_product_id', 'product_name', 'category', 'price', 'cost', 'product_description']
df_fixed columns: ['order_id', 'customer_id', 'order_date', 'quantity', 'line_revenue', 'product_id', 'product_name', 'category', 'price', 'cost', 'product_description']
df_orders_large columns: ['order_id', 'customer_id', 'product_id', 'order_date', 'quantity', 'line_revenue', 'skewed_product_id', 'salt', 'salted_key', 'product_id', 'product_name', 'category', 'price', 'cost', 'product_description', 'salt', 'salted_key', 'dup_key']


###### **<mark>STEP 1 — CREATE MULTIPLE VERSIONS - for Time Travel</mark>**

In [77]:
spark.sql("DROP TABLE IF EXISTS orders_demo")

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 79, Finished, Available, Finished, False)

DataFrame[]

In [78]:
df_fixed.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("orders_demo")

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 80, Finished, Available, Finished, False)

In [86]:
df_fixed.write.format("delta") \
    .mode("append") \
    .saveAsTable("orders_demo")

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 88, Finished, Available, Finished, False)

In [87]:
spark.sql("DESCRIBE HISTORY orders_demo").show(truncate=False)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 89, Finished, Available, Finished, False)

+-------+-----------------------+------+--------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+-------------------------------------------------------------------------+------------+-------------------------------------------------------------+
|version|timestamp              |userId|userName|operation                        |operationParameters                                                                                                                                                     |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                         |userMetadata|engineInfo                                                   |
+-------+-----------------------+------+--------+-----------------

###### **<mark>Checking each version count value as Proof of concept, the count varies in each version</mark>**

In [88]:
spark.read.format("delta").option("versionAsOf", 0).table("orders_demo").count()

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 90, Finished, Available, Finished, False)

24000000

In [89]:
spark.read.format("delta").option("versionAsOf", 1).table("orders_demo").count()

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 91, Finished, Available, Finished, False)

48000000

In [90]:
spark.read.format("delta").option("versionAsOf", 2).table("orders_demo").count()

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 92, Finished, Available, Finished, False)

72000000

In [91]:
spark.read.format("delta").option("versionAsOf", 8).table("orders_demo").count()

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 93, Finished, Available, Finished, False)

216000000

In [92]:
for v in [0, 1, 2, 8]:
    count_v = spark.read.format("delta").option("versionAsOf", v).table("orders_demo").count()
    print(f"Version {v} count = {count_v}")

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 94, Finished, Available, Finished, False)

Version 0 count = 24000000
Version 1 count = 48000000
Version 2 count = 72000000
Version 8 count = 216000000


In [93]:
spark.sql("DESCRIBE HISTORY orders_demo").show(truncate=False)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 95, Finished, Available, Finished, False)

+-------+-----------------------+------+--------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+-------------------------------------------------------------------------+------------+-------------------------------------------------------------+
|version|timestamp              |userId|userName|operation                        |operationParameters                                                                                                                                                     |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                         |userMetadata|engineInfo                                                   |
+-------+-----------------------+------+--------+-----------------

In [94]:
spark.sql("VACUUM orders_demo RETAIN 168 HOURS").show(truncate=False)

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 96, Finished, Available, Finished, False)

+-----------------------------------------------------------------------------------------------------------------------------------------+
|path                                                                                                                                     |
+-----------------------------------------------------------------------------------------------------------------------------------------+
|abfss://eaf0c26f-0ff8-484b-907a-fbe7a597851a@onelake.dfs.fabric.microsoft.com/8702ce55-bae0-44d5-b9b0-0018d638c4c0/Tables/dbo/orders_demo|
+-----------------------------------------------------------------------------------------------------------------------------------------+



In [95]:
for v in [0, 1, 2, 8]:
    count_v = spark.read.format("delta").option("versionAsOf", v).table("orders_demo").count()
    print(f"Version {v} count = {count_v}")

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 97, Finished, Available, Finished, False)

Version 0 count = 24000000
Version 1 count = 48000000
Version 2 count = 72000000
Version 8 count = 216000000


In [96]:
for v in [0, 1, 2, 8]:
    count_v = spark.read.format("delta").option("versionAsOf", v).table("orders_demo").count()
    print(f"Version {v} count = {count_v}")

StatementMeta(, 4e902950-0b7f-4f61-9d86-21e8f28ff603, 98, Finished, Available, Finished, False)

Version 0 count = 24000000
Version 1 count = 48000000
Version 2 count = 72000000
Version 8 count = 216000000


In [1]:
spark.sql("""
OPTIMIZE orders_demo
ZORDER BY (customer_id)
""")

StatementMeta(, faa9b301-47fd-4c75-8b8b-f48358ed9144, 3, Finished, Available, Finished, False)

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,numFilesUpdatedWithoutRewrite:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesUpdatedWithoutRewrite:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemovedBreakdown:array<struct<reason:string,metrics:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>>>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,

In [2]:
spark.sql("""
SELECT * FROM orders_demo WHERE customer_id = 123
""").show()

StatementMeta(, faa9b301-47fd-4c75-8b8b-f48358ed9144, 4, Finished, Available, Finished, False)

+--------+-----------+----------+--------+------------------+----------+------------+-----------+------+------+--------------------+
|order_id|customer_id|order_date|quantity|      line_revenue|product_id|product_name|   category| price|  cost| product_description|
+--------+-----------+----------+--------+------------------+----------+------------+-----------+------+------+--------------------+
| 5962206|        123|2025-01-29|       2|             33.08|     21262|         say|Electronics|368.13| 47.58|Animal doctor ton...|
| 5349076|        123|2025-11-27|       5|            1706.9|     23321|       night|      Books| 17.91| 40.71|Commercial alread...|
|   25606|        123|2025-04-19|       4|            778.28|     28398|     feeling|   Clothing|178.98| 44.25|Street blood Demo...|
|   25606|        123|2025-04-19|       5|             303.2|      3010|      couple|       Home| 85.84| 42.48|Himself family ro...|
|  804115|        123|2026-01-23|       5|1281.3999999999999|       8

In [3]:
spark.sql("""
EXPLAIN SELECT * FROM orders_demo WHERE customer_id = 123
""").show(truncate=False)

StatementMeta(, faa9b301-47fd-4c75-8b8b-f48358ed9144, 5, Finished, Available, Finished, False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|plan                                                                                                                                                                                                                          